In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======================================
# 対象フォルダを指定
# ======================================
target = "outputs/experiments/transformer_grf_single_weighted_std_20260610_150200"

# ======================================
# フォルダ存在確認
# ======================================
if not os.path.exists(target):
    raise FileNotFoundError(f"Directory not found: {target}")

print(f"Target Directory: {target}")

weights_all = []
weight_type = None
target_type = None

# ======================================
# 各foldのloss weightを読み込み
# ======================================
for fold in range(1, 7):
    file_path = os.path.join(target, f"loss_weights_fold{fold}.json")

    if not os.path.exists(file_path):
        print(f"Warning: File not found: {file_path}")
        continue

    with open(file_path, "r") as f:
        data = json.load(f)

    weights_all.append(data["weights"])

    if weight_type is None:
        weight_type = data.get("weight_type")

    if target_type is None:
        target_type = data.get("target_type")

if not weights_all:
    raise FileNotFoundError(
        "No loss_weights_fold*.json files found in this directory."
    )

# ======================================
# 平均・標準偏差計算
# ======================================
weights_arr = np.array(weights_all)

mean_weights = np.mean(weights_arr, axis=0)
std_weights = np.std(weights_arr, axis=0)

# ======================================
# target名の読み込み
# ======================================
target_names = []

target_names_path = os.path.join(target, "target_names.json")

if os.path.exists(target_names_path):
    with open(target_names_path, "r") as f:
        target_names = json.load(f)
else:
    target_names = [f"Feature {i}" for i in range(len(mean_weights))]

# ======================================
# DataFrame作成
# ======================================
df = pd.DataFrame({
    "Target": target_names,
    "Mean Weight": mean_weights,
    "Std Weight": std_weights
})

# 小数点6桁表示
df["Mean Weight"] = df["Mean Weight"].round(6)
df["Std Weight"] = df["Std Weight"].round(6)

# ======================================
# サマリー表示
# ======================================
print("\n" + "=" * 55)
print("  Loss Weights Analysis Summary")
print("=" * 55)
print(f"Target Type   : {target_type}")
print(f"Weight Type   : {weight_type}")
print(f"Analyzed Folds: {len(weights_all)} / 6")
print("-" * 55)

display(df)

print("-" * 55)
print("Raw Mean Weights:")
print(list(mean_weights.round(6)))
print("=" * 55)

# ======================================
# 棒グラフ
# ======================================
plt.figure(figsize=(8, 4))
plt.bar(df["Target"], df["Mean Weight"])
plt.ylabel("Mean Weight")
plt.xlabel("Target")
plt.title("Average Loss Weights Across Folds")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()